# Implementation on the FPGA-based system

$\renewcommand{\ket}[1]{\left|#1\right\rangle}\renewcommand{\bra}[1]{\left\langle #1\right|}\renewcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle}\renewcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}$

This notebook explains how the FPGA timing estimates for the classical MCMC proposal kernels are obtained.

The FPGA implementation is the closest classical analogue of the quantum circuit implementation among the classical backends considered here. The computation is implemented as a fixed combinational datapath for each system size, rather than as a sequence of instructions executed by a CPU or GPU through a memory hierarchy. In general, the amount of hardware grows with the system size, while the critical-path latency grows much more weakly.

We used the AMD Vitis/Vivado toolchain and implemented the kernels in C++ HLS (High-Level Synthesis). The source code is not ordinary CPU C++ in the performance sense. It is hardware-oriented C++: arrays, loops, and helper functions are interpreted as hardware structures after applying HLS directives. This allows rapid prototyping and testing of the arithmetic logic in C++, while the synthesis to an RTL-level representation such as Verilog or VHDL is performed automatically later.

**Table of contents**
1. Source
2. Algorithm
3. How to estimate the timing
4. Reproducibility

## Source

Both the code and the data are available in the `estimation_timing/fpga` folder.

The folder contains:

* folder `src`: contains the files `local_move_hls.cpp` and `uniform_move_hls.cpp`, which implement the circuits estimating the acceptance probability of the local and uniform trial moves. These sources require the Chebyshev coefficients used to implement the exponential acceptance-probability function. The coefficients are provided in the file `exp_cheby.hpp`, which is generated automatically by the script `make_exp_cheby.py`.
* folder `tb`: contains the testbench files. The file `tb_local_move_n10_all.cpp` instantiates a single Ising model with $n=10$ spins and checks that the circuit output matches the reference implementation for all $2^{10}$ input states and all $10$ possible local flips. The file `tb_uniform_move_n10_all.cpp` performs the corresponding check for the uniform move by scanning all $2^{10}$ old states and a deterministic set of $10$ proposed new states per old state.
* folder `scripts`: contains the Tcl directives to run synthesis in `run_synthesis_move.tcl`, and the Tcl directives to run the testbenches in `run_csim_local_n10.tcl` and `run_csim_uniform_n10.tcl`. Tcl is the scripting language used by the AMD toolchain.
* folder `logs`: contains the logs of the executed scripts.
* folder `reports`: contains the synthesis reports for the local and uniform circuits for $n=8,16,32,64$. These reports contain the latency and resource estimates used below.
* `launch_testbench_local_n10.sh`: launches the local-move testbench and checks for errors.
* `launch_testbench_uniform_n10.sh`: launches the uniform-move testbench and checks for errors.
* `launch_synthesis_many_n.sh`: launches the synthesis of both moves for $n=8,16,32,64$.
* `copy_synthesis_reports.sh`: copies the synthesis reports into the `reports` folder from the implementation folders generated by synthesis, which are too large to keep.
* `launch_visualization.sh`: generates a reduced mock implementation of the circuits for $n=8$ and $8$ bits of fixed-point precision for visualization purposes.
* folder `visualization`: contains the reduced synthesized implementation used to inspect the generated Verilog and to generate figures with Yosys. The raw visualizations are not very readable, so I recreated the architecture diagrams manually in TikZ. Both the LaTeX source and the PDF are included in the folder.


## Algorithm

### FPGA local move implementation

The local FPGA kernel implements the Metropolis acceptance logic for a single-spin-flip proposal. The input spin configuration is fixed, and a one-hot mask selects the spin $i$ to flip. Since only one spin changes, the circuit does not recompute the full SK energy. Instead, it uses the local identity $\Delta E_i/\alpha = 2s_i(h_i/\alpha+\sum_{j\neq i}(J_{ij}/\alpha)s_j)$.

The coefficients $h_i$ and $J_{ij}$ enter the circuit already normalized by $\alpha$. This keeps the local field, $\Delta E_i/\alpha$, and $x=\max(\Delta E_i/\alpha,0)$ in a fixed-point format with only one integer/sign bit. The sum over $j\neq i$ is implemented as a parallel adder tree. The terms $(J_{ij}/\alpha)s_j$ are generated in parallel and reduced with logarithmic depth, so the summation depth scales as $O(\log N)$ rather than $O(N)$. This explains why latency grows only weakly with $N$: most of the cost is paid in hardware area.

The Metropolis cut is then applied by setting $x=\max(\Delta E_i/\alpha,0)$. Downhill moves have $x=0$ and are accepted with probability one, while uphill moves use the factor $\exp(-\beta\alpha x)$. This exponential is approximated by a truncated Chebyshev series and evaluated with the Clenshaw algorithm. The polynomial is not evaluated globally on $[0,1]$: the generated header defines a cutoff $X_{\rm cut}$, maps $x\in[0,X_{\rm cut}]$ to the Chebyshev variable $y\in[-1,1]$, and returns zero for $x\ge X_{\rm cut}$. Clenshaw evaluates $\sum_k c_kT_k(y)$ through a backward recurrence, avoiding the explicit construction of Chebyshev polynomials.

The selected clock period is mainly limited by the fixed-point multiplications in the polynomial block. A higher frequency could be targeted by pipelining or splitting these multiplications into multi-cycle operations, but this would increase the latency in clock cycles.

**Datatypes and constants.**

- `SK_N`: compile-time number of spins, denoted by $n$. In the visualization, `SK_N = 8`, hence $n=8$.
- `FRAC`: compile-time number of fractional bits, denoted by $F$. In the visualization, `FRAC = 8`, hence $F=8$.
- `NUM_J`: number of independent SK couplings $J_{ij}$ with $i<j$. In general, $\mathrm{NUM\_J}=n(n-1)/2$. For $n=8$, $\mathrm{NUM\_J}=8\cdot 7/2=28$.
- `coeff_t`: signed fixed point $\langle 1.F\rangle$. It stores the normalized coefficients $h_i/\alpha$ and $J_{ij}/\alpha$.
- `delta_t`: signed fixed point $\langle 1.F\rangle$. It stores normalized local fields, normalized energy differences $\Delta E/\alpha$, and the clipped value $x=\max(\Delta E/\alpha,0)$. With the chosen normalization, these quantities fit in a signed format with one integer/sign bit.
- `prob_t`: unsigned fixed point $\langle 1.31\rangle$. It stores the Metropolis acceptance probability in $[0,1]$.
- `poly_t`: signed fixed point $\langle 4.36\rangle$. It stores Chebyshev coefficients and intermediate values in the Clenshaw recurrence.
- `scale_t`: unsigned fixed point $\langle 16.32\rangle$. It stores $X_{\rm cut}^{-1}=\beta\alpha/\tau$, which can be larger than one.
- `spins`: bitstring $\langle n\rangle$. Each bit represents one Ising spin, with $0\mapsto -1$ and $1\mapsto +1$.
- `flip_mask`: bitstring $\langle n\rangle$. It is one-hot and selects the spin $i$ to flip.
- `EXP_X_CUT`: fixed cutoff $X_{\rm cut}$ generated offline in `exp_cheby.hpp`.
- `EXP_INV_X_CUT`: fixed scale $X_{\rm cut}^{-1}$ generated offline in `exp_cheby.hpp`.
- `CHEB_COEFFS`: fixed Chebyshev coefficients generated offline in `exp_cheby.hpp`.

**Inputs.**

- `h`: vector of $n$ `coeff_t` values. It stores the normalized fields $h_i/\alpha$.
- `J_edges`: vector of `NUM_J` `coeff_t` values. It stores the normalized couplings $J_{ij}/\alpha$ in packed upper-triangular form.
- `spins`: bitstring $\langle n\rangle$ representing the current spin configuration.
- `flip_mask`: one-hot bitstring $\langle n\rangle$ selecting the spin to flip.

**Outputs.**

- `delta_x`: pointer to a `delta_t` value containing $x=\max(\Delta E_i/\alpha,0)$.
- `accept_prob`: pointer to a `prob_t` value containing the approximation to $\exp(-\beta\alpha x)$.

**Methods.**

- `spin_value`: converts a spin bit into an Ising value in $\{-1,+1\}$.
- `edge_index`: maps a pair $(i,j)$ to the packed upper-triangular index used to access $J_{ij}$.
- `selected_field`: uses `flip_mask` as a one-hot multiplexer to select $h_i$.
- `selected_coupling_for_j`: uses `flip_mask` as a one-hot multiplexer to select each $J_{ij}$ for the chosen flipped spin.
- `selected_spin`: helper that extracts the selected spin $s_i$ from `spins` and `flip_mask`.
- `tree_sum<>`: implements a fully unrolled binary adder tree for the local field terms. In the local circuit, it reduces one field term and $n$ coupling slots; the slot with $j=i$ is zero.
- `local_delta_over_alpha`: computes $\Delta E_i/\alpha = 2s_i(h_i/\alpha+\sum_{j\neq i}(J_{ij}/\alpha)s_j)$.
- `positive_part`: computes $x=\max(\Delta E_i/\alpha,0)$. Downhill moves therefore have $x=0$ and acceptance probability $1$.
- `chebyshev_g`: evaluates the piecewise Chebyshev approximation of $\exp(-\beta\alpha x)$ using the Clenshaw recurrence.
- `local_spin_flip_operation`: top-level HLS function combining the local delta computation, clipping, and probability approximation.

![Schematic representation of the datapath for the local move circuit](../estimation_timing/fpga/visualization/schematics/local_N8_cpp_architecture_tikz.png)

The figure shows the local-move datapath for $n=8$ and $F=8$. Blue boxes denote top-level inputs and outputs, green boxes denote helper blocks, and red boxes denote the main arithmetic blocks. A solid arrow denotes runtime data flow: the output of one block feeds the input of another block. The local circuit first selects the relevant $h_i$, $J_{ij}$, and $s_i$, then forms the local-field terms. A log-depth adder tree computes the local field, after which the circuit forms $2s_i\,\mathrm{field}_i/\alpha$, clips it to $x=\max(\Delta E_i/\alpha,0)$, and feeds $x$ to the piecewise Chebyshev acceptance-probability block.


### FPGA uniform move implementation

The uniform FPGA kernel implements the Metropolis acceptance logic for a dense non-local proposal. In this case, the proposed spin configuration can differ from the old configuration at many spin positions, so the local single-spin-flip identity no longer applies. The circuit must therefore compute the full dense SK energy of the proposed configuration, $E_{\rm new}/\alpha = -\sum_i (h_i/\alpha)s_i - \sum_{i<j}(J_{ij}/\alpha)s_i s_j$, then subtract the supplied old energy $E_{\rm old}/\alpha$.

As in the local circuit, the coefficients enter already normalized by $\alpha$. With the chosen normalization, the full normalized energy $E/\alpha$, the normalized energy difference $(E_{\rm new}-E_{\rm old})/\alpha$, and the partial sums used by the dense energy computation fit in the same signed one-integer-bit format.

**Datatypes and constants.**

- `SK_N`: compile-time number of spins, denoted by $n$. In the visualization, `SK_N = 8`, hence $n=8$.
- `FRAC`: compile-time number of fractional bits, denoted by $F$. In the visualization, `FRAC = 8`, hence $F=8$.
- `NUM_J`: number of independent SK couplings $J_{ij}$ with $i<j$. In general, $\mathrm{NUM\_J}=n(n-1)/2$. For $n=8$, $\mathrm{NUM\_J}=28$.
- `NUM_ENERGY_TERMS`: number of terms in the full dense energy. In general, $\mathrm{NUM\_ENERGY\_TERMS}=n+\mathrm{NUM\_J}$. For $n=8$, $\mathrm{NUM\_ENERGY\_TERMS}=8+28=36$.
- `coeff_t`: signed fixed point $\langle 1.F\rangle$. It stores the normalized coefficients $h_i/\alpha$ and $J_{ij}/\alpha$.
- `delta_t`: signed fixed point $\langle 1.F\rangle$. It stores normalized energies $E/\alpha$, normalized energy differences $(E_{\rm new}-E_{\rm old})/\alpha$, intermediate partial sums, and the clipped value $x$.
- `prob_t`: unsigned fixed point $\langle 1.31\rangle$. It stores the acceptance probability in $[0,1]$.
- `poly_t`: signed fixed point $\langle 4.36\rangle$. It stores Chebyshev coefficients and Clenshaw intermediate values.
- `scale_t`: unsigned fixed point $\langle 16.32\rangle$. It stores $X_{\rm cut}^{-1}=\beta\alpha/\tau$.
- `new_spins`: bitstring $\langle n\rangle$. It represents the proposed new spin configuration.
- `old_energy`: signed fixed point $\langle 1.F\rangle$. It stores the already-known normalized energy $E_{\rm old}/\alpha$.
- `EXP_X_CUT`: fixed cutoff $X_{\rm cut}$ generated offline in `exp_cheby.hpp`.
- `EXP_INV_X_CUT`: fixed scale $X_{\rm cut}^{-1}$ generated offline in `exp_cheby.hpp`.
- `CHEB_COEFFS`: fixed Chebyshev coefficients generated offline in `exp_cheby.hpp`.

**Inputs.**

- `h`: vector of $n$ `coeff_t` values. It stores the normalized fields $h_i/\alpha$.
- `J_edges`: vector of `NUM_J` `coeff_t` values. It stores the normalized couplings $J_{ij}/\alpha$ in packed upper-triangular form.
- `new_spins`: bitstring $\langle n\rangle$ representing the proposed new spin configuration.
- `old_energy`: `delta_t` value containing the already-known normalized old energy $E_{\rm old}/\alpha$.

**Outputs.**

- `new_energy`: pointer to a `delta_t` value containing the computed normalized proposed energy $E_{\rm new}/\alpha$.
- `delta_x`: pointer to a `delta_t` value containing $x=\max((E_{\rm new}-E_{\rm old})/\alpha,0)$.
- `accept_prob`: pointer to a `prob_t` value containing the approximation to $\exp(-\beta\alpha x)$.

**Methods.**

- `spin_value`: converts each bit of `new_spins` into an Ising spin in $\{-1,+1\}$.
- `edge_index`: maps each pair $(i,j)$ to the packed upper-triangular index used to access $J_{ij}$.
- `field_delta_term`: computes each field contribution $-h_i s_i$.
- `pair_delta_term`: computes each pair contribution $-J_{ij}s_i s_j$.
- `tree_sum<>`: implements a fully unrolled binary adder tree over all dense energy terms. For $n=8$, there are $8$ field terms and $28$ pair terms, hence $36$ total terms.
- `dense_energy_over_alpha`: computes the full proposed normalized energy $E_{\rm new}/\alpha = -\sum_i (h_i/\alpha)s_i - \sum_{i<j}(J_{ij}/\alpha)s_i s_j$.
- `positive_part`: computes $x=\max((E_{\rm new}-E_{\rm old})/\alpha,0)$.
- `chebyshev_g`: evaluates the same piecewise Chebyshev approximation of $\exp(-\beta\alpha x)$ used by the local circuit.
- `uniform_move_operation`: top-level HLS function computing the dense proposed energy, subtracting the old energy, clipping the difference, and evaluating the acceptance probability.

![Schematic representation of the datapath for the uniform move circuit](../estimation_timing/fpga/visualization/schematics/uniform_N8_cpp_architecture_tikz.png)

The figure shows the uniform-move datapath for $n=8$ and $F=8$. The color convention matches the local circuit: blue boxes denote top-level inputs and outputs, green boxes denote helper blocks, and red boxes denote main arithmetic blocks. Solid arrows denote runtime data flow. Unlike the local move, the uniform move cannot use the single-spin-flip formula. It computes all field terms $-h_i s_i$ and all pair terms $-J_{ij}s_i s_j$, reduces them through a log-depth adder tree, subtracts the supplied old energy, clips the result to $x$, and evaluates the Metropolis acceptance probability through the shared piecewise Chebyshev block.


## How to estimate the timing

The timing information is extracted from the Vitis HLS synthesis reports in the `reports/` folder. The relevant files are the `local_spin_flip_operation_csynth*` reports for the local move and the `uniform_move_operation_csynth*` reports for the uniform move. Vitis emits these reports in both human-readable and machine-readable formats.

The selected clock period is `3.00 ns`, corresponding to a target frequency of about `333 MHz`. This value is close to the practical limit of the current implementation because the critical path is bottlenecked by wide fixed-point multiplications, especially in the piecewise Chebyshev block. A higher target frequency could be chosen if these multiplications were split into pipelined or multi-cycle operators, but this would increase the number of latency cycles.

For the local move, the arithmetic cost is the local energy difference, involving one selected field term and `N-1` coupling terms, hence `N` local delta-energy terms up to the zero self-coupling slot used in the implementation. For the uniform move, the arithmetic cost is the full dense SK energy, involving `N` field terms and `N(N-1)/2` coupling terms, hence `N(N+1)/2` dense energy terms.

The attached XML report is one example of the data used below: for the local kernel at `N=8` and `FRAC=24`, it reports a target clock of `3.00 ns`, latency `91` cycles, real-time latency `0.273 us`, and resources `50` DSP, `6,963` FF, and `11,132` LUT. This matches the first row of the local-move table.

**Local move: local delta energy**

| N | Fractional bits | Local delta energy terms | Target clock (ns) | Latency (cycles) | Latency (us) | DSP | FF | LUT |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 8  | 24 | 8  | 3.00 | 91 | 0.273 | 50  | 6,963  | 11,132 |
| 16 | 28 | 16 | 3.00 | 92 | 0.276 | 146 | 11,719 | 21,318 |
| 32 | 31 | 32 | 3.00 | 92 | 0.276 | 274 | 20,974 | 54,540 |
| 64 | 35 | 64 | 3.00 | 93 | 0.279 | 530 | 47,471 | 190,598 |

**Uniform move: dense energy**

| N | Fractional bits | Dense energy terms | Target clock (ns) | Latency (cycles) | Latency (us) | DSP | FF | LUT |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 8  | 24 | 36   | 3.00 | 89 | 0.267 | 18 | 4,975   | 16,728 |
| 16 | 28 | 136  | 3.00 | 90 | 0.270 | 18 | 9,436   | 48,876 |
| 32 | 31 | 528  | 3.00 | 92 | 0.276 | 18 | 28,128  | 183,801 |
| 64 | 35 | 2080 | 3.00 | 93 | 0.279 | 18 | 111,827 | 778,597 |

The main observation is that the latency in cycles is almost constant across the tested sizes. This is because the FPGA implementation exposes a large amount of spatial parallelism: many arithmetic terms are synthesized as parallel hardware rather than being evaluated sequentially as in a CPU loop. Therefore, increasing `N` mostly increases resource usage, especially LUTs and registers, rather than increasing latency proportionally.

The difference between the two proposals is clearer in the resource scaling. The local move uses `O(N)` local delta-energy terms, while the uniform move uses `O(N^2)` dense energy terms. This is reflected in the LUT growth: for the uniform move the LUT count grows from `16,728` at `N=8` to `778,597` at `N=64`, while for the local move it grows from `11,132` to `190,598`. The nearly flat latency should therefore not be interpreted as the arithmetic cost being independent of `N`. Instead, it means that the cost is paid mostly in hardware area, with quadratic resource scaling for the dense uniform proposal.

The following model captures the weak latency scaling with the number of spins.


In [1]:
import numpy as np

N = np.array([8, 16, 32, 64], dtype=float)
local_cycles = np.array([91, 92, 92, 93])
uniform_cycles = np.array([89, 90, 92, 93])

def fit_A_B_log2n(n, y):
    X = np.column_stack([np.ones_like(n), np.log2(n)])
    A, B = np.linalg.lstsq(X, y, rcond=None)[0]
    yfit = X @ np.array([A, B])
    return A, B, yfit

A_local, B_local, yfit_local = fit_A_B_log2n(N, local_cycles)
A_uniform, B_uniform, yfit_uniform = fit_A_B_log2n(N, uniform_cycles)

print(f"local:   cycles(N) = {A_local:.3f} + {B_local:.3f} log2(N)")
print(f"uniform: cycles(N) = {A_uniform:.3f} + {B_uniform:.3f} log2(N)")
print(f"local:   latency_us(N) = {3e-3*A_local:.6f} + {3e-3*B_local:.6f} log2(N) [μs]")
print(f"uniform: latency_us(N) = {3e-3*A_uniform:.6f} + {3e-3*B_uniform:.6f} log2(N) [μs]")


local:   cycles(N) = 89.300 + 0.600 log2(N)
uniform: cycles(N) = 84.700 + 1.400 log2(N)
local:   latency_us(N) = 0.267900 + 0.001800 log2(N) [μs]
uniform: latency_us(N) = 0.254100 + 0.004200 log2(N) [μs]


### Reproducibility

The FPGA/HLS workflow uses AMD Vitis/Vivado 2025.2. I downloaded the AMD unified installer, made it executable, extracted it with `--noexec`, and generated the batch-install configuration with:

```bash
chmod +x FPGAs_AdaptiveSoCs_Unified_*_Lin64.bin
./FPGAs_AdaptiveSoCs_Unified_SDI_2025.2_1114_2157_Lin64.bin --noexec --target <installer extraction dir>
cd <installer extraction dir>
./xsetup -b ConfigGen
```

Before installing, edit the installer configuration file:

```bash
~/.Xilinx/install_config.txt
```

I set the relevant fields as follows:

```bash
# Path where AMD FPGAs & Adaptive SoCs software will be installed.
Destination=<install root>/xlnx
# Choose the Products/Devices that you would like to install.
Modules=Virtex UltraScale+ FPGAs:1
```

Then I launched the final batch installation with:

```bash
./xsetup --agree XilinxEULA,3rdPartyEULA --batch Install --config ~/.Xilinx/install_config.txt
```

Each run sources Vitis with `nounset` disabled:

```bash
set +u
source <install root>/xlnx/2025.2/Vitis/settings64.sh
set -u
```

This avoids failures when AMD setup scripts reference initially unset variables such as `PYTHONPATH`. Check the installation with:

```bash
which vitis-run
which vivado
vitis --version
vivado -version
```

The workflow also uses Python to generate the Chebyshev coefficients for the piecewise approximation of the Metropolis acceptance factor. I used Python 3.11 and `pip`; change the Python executable name in the scripts if your system uses a different one.

```bash
python3.11 -m pip install numpy scipy
```

I ran the local and uniform C-simulation testbenches with:

```bash
./launch_testbench_local_n10.sh > logs/log_testbench_local_n10.txt
./launch_testbench_uniform_n10.sh > logs/log_testbench_uniform_n10.txt
```

You need to change the references to your own Vitis installation path and Python executable if they differ. The local testbench fixes `SK_N=10`, scans all $2^{10}$ spin configurations, and tests all ten one-hot local spin flips per configuration. It checks `delta_x` against an independent floating-point reference for $\max(\Delta E_i/\alpha,0)$ and checks `accept_prob` against `std::exp(-TEST_LAMBDA * delta_x)`. The uniform testbench also fixes `SK_N=10`. It scans all $2^{10}$ old configurations and a deterministic set of ten proposed new configurations per old state. It checks `new_energy`, `delta_x`, and `accept_prob` against independent dense-energy and Metropolis references.

I ran synthesis with:

```bash
./launch_synthesis_many_n.sh > logs/log_launch_synthesis_many_n.txt
```

The synthesis launcher scans $N = 8,16,32,64$ and synthesizes both HLS top functions: `local_spin_flip_operation` and `uniform_move_operation`. For each $N$, the script computes the required number of fractional fixed-point bits $F$, regenerates `src/exp_cheby.hpp`, and calls `scripts/run_synthesis_move.tcl`. The synthesis targets the `xcvu19p-fsva3824-2-e` FPGA with clock period `3.000 ns`, corresponding to a nominal `333.3 MHz` clock.

After synthesis, I collected the reports with:

```bash
./copy_synthesis_reports.sh
```

The script copies the synthesis reports into the `reports/` folder.

To generate a diagrammatic representation of the RTL code with Yosys, run:

```bash
./launch_visualization.sh > logs/log_launch_visualization.txt
```

This creates a reduced `N=8`, `FRAC=8` visualization build. The script copies the HLS sources into `visualization/temp`, replaces major `#pragma HLS INLINE` directives with `#pragma HLS INLINE off`, and extracts hierarchy-preserving Verilog into:

```bash
visualization/hls_local_spin_flip_operation_N8_FRAC8_verilog/
visualization/hls_uniform_move_operation_N8_FRAC8_verilog/
```

Disabling inlining makes the generated RTL less optimized but easier to inspect, because HLS preserves more function-level module boundaries. The visualization script can also generate schematic images if Yosys, Graphviz, and `rsvg-convert` are available:

```bash
sudo apt-get install yosys graphviz librsvg2-bin
```

In that case, the script saves the generated schematic images in `visualization/schematics/`. I also created manually written LaTeX/TikZ circuit diagrams in the same visualization folder. These diagrams give a cleaner architectural representation than the raw RTL schematics.
